# Spatial perturbation benchmark — the phase-1 pipeline

Evaluates how well a model predicts the **spatial effect of a CRISPR perturbation**: when a gene is knocked out, how does the perturbed cell itself change, and how does that change spread to its spatial neighbours? The same cell is never measured both perturbed and unperturbed, so everything is scored at the **distribution level** against a **control reference**. Run on the lab server (env `spatial-tumor-ai`), where the Saunders `.h5mu` slices and the offline scGEN dumps live.

## 0. What's new in this refactor (read this first)

**1. Two-layer adapters.** An *input* adapter (`SaundersAdapter`) normalises a raw `.h5mu` slice into one `StandardData`. An *output* adapter (`adapters.internal_output.to_prediction`, `models.scgen_model.ScgenSeedModel`) normalises every model's native output. Models never touch each other's formats.

**2. The 3-dim output contract `{D1, D2, D3}`** (`spbench.prediction.StandardPrediction`). Every prediction is expressed as up to three dims, and a model fills only the ones it covers (`.covered_dims()` / `.as_dict()`):
- **D1 — self**: the perturbed cell's OWN expression shift (the seed).
- **D2 — niche expression**: the aggregated bystander-neighbour expression shift.
- **D3 — niche composition**: the neighbourhood cell-type composition (a simplex).
This is the **capability matrix**: scGEN is a D1-only model; the Gaussian/GCN propagation baselines cover D2; D3 comes from the niche module. A model is only ever scored on the dims it actually covers.

**3. Sample-level aggregate control** (`reference_aggregate.aggregate_control` -> `harness._control_reference_aggregate`). The reference 'unperturbed' state is now a per-cell-type aggregate over CONTROL cells only — no nearest-neighbour matched-control pairing in feature space, so no matched-control leakage. The SAME aggregate-control reference now drives EVERY comparison: the D1 seed baseline (`seed_ref`) and the D2 niche no-effect baseline (`e_null` = the average same-type control cell's neighbourhood) both come from `reference_aggregate.control_reference_centers` — the old `match_reference_centers` expression-nearest-neighbour matching is fully retired. `run_benchmark` uses this by default.

**4. `eval_X` — one unified scoring space.** PCC-delta is not cross-space robust, so the prediction, the observed cells, and the delta baseline must all be compared in the SAME space. For scGEN the natural space is its log-norm training space: we pass `eval_X=lognorm_X` (the `build_lognorm_X` matrix) into `fill_2x2`, which slices the observed/reference seed cells into that same matrix so scGEN's D1 is scored fairly.

**5. MC-spatial quadrant x dimension stratification.** MC-spatial flags, per guide, whether a real **Self** (X) and/or **Niche** (Y) effect exists, giving four quadrants: **Both**, **X-Only (Self)**, **Y-Only (Niche)**, **Inert**. We score **D1 only where there is Self signal** (X-Only/Both), **D2 only where there is Niche signal** (Y-Only/Both), and use the **Inert** quadrant as a rigorous **negative control** (a good model should predict ~no effect there). This avoids diluting a niche score over perturbations that have no niche signal.

## 1. Config
All server-specific paths live here, at the top, so a server run only edits this one cell. `SLICE_DIR` is the Saunders directory; `MAX_FILES` how many slices to pool. The two OPTIONAL inputs — `SCGEN_DUMP_DIR` (offline `{P}_seed.h5ad` dumps) and `DUAL_METRICS_CSV` (MC-spatial `*_Dual_Metrics.csv`) — may be left as `None`/missing; the notebook then runs baseline-only / unstratified and says so explicitly.

In [ ]:
from pathlib import Path

# --- required: the Saunders slice(s) ---
SLICE_DIR = '/home/yiru/database/spatial_perturbed_processed/CRISPR_based/Saunders_2025_40513557'
MAX_FILES = 1                      # pool this many .h5mu slices (1 = a single slice)
COUNTS_LAYER = 'X'                 # raw-count layer for scGEN log-norm export (mod/rna/X)

# --- optional input A: offline scGEN seed dumps ({P}_seed.h5ad), one per perturbation ---
SCGEN_DUMP_DIR = None             # e.g. '/home/yiru/scgen_dumps/Saunders_slice0'; None -> skip scGEN

# --- optional input B: MC-spatial four-quadrant labels for this slice ---
DUAL_METRICS_CSV = None           # e.g. '/home/yiru/mc_spatial_out/slice0_Dual_Metrics.csv'; None -> unstratified

# --- evaluation perturbation list (MC-spatial Self/Niche-significant guides for this slice) ---
EVAL = ['Pck1','Rrbp1','Hspd1','Psmc1','Sepp1','Bcl2l1','Vcp',
        'Ass1','Pten','Rrn3','Letm1','Hspa5','Sec61b','Rngtt']

K_GRAPH, K_REF, NICHE_HOPS = 15, 5, 1
print('SLICE_DIR        =', SLICE_DIR)
print('SCGEN_DUMP_DIR   =', SCGEN_DUMP_DIR, '(scGEN D1 will run only if these dumps exist)')
print('DUAL_METRICS_CSV =', DUAL_METRICS_CSV, '(quadrant stratification only if present)')

## 1b. Imports

In [ ]:
import matplotlib
%matplotlib inline
import numpy as np, pandas as pd

from spbench.adapters.saunders import SaundersAdapter
from spbench.adapters.scgen_export import build_lognorm_X
from spbench.adapters.internal_output import to_prediction
from spbench.prediction import StandardPrediction
from spbench.graph import build_knn_graph
from spbench.niche import build_spatial_graph, compute_niche_composition
from spbench.reference_aggregate import aggregate_control
from spbench.config import run_benchmark
from spbench.harness import (fill_2x2, _control_reference_aggregate, _control_residuals)
from spbench.compare import evaluate_seed
from spbench.models.trivial_seed import TrivialSeed
from spbench.models.gaussian_prop import GaussianProp
from spbench.models.gcn_prop import SimpleGCN
from spbench import mc_spatial_join as mcj
from spbench import mc_spatial_report as mcr

## 2. Load the data
`SaundersAdapter(SLICE_DIR, max_files=MAX_FILES, counts_layer='X').load()` reads the raw `.h5mu` slice(s) and normalises them into one `StandardData`: `data.X` is the (z-scored) `layers/raw_scaled` matrix; `counts_layer='X'` ALSO loads the raw integer counts into `data.meta['counts']` (needed for scGEN's log-norm space in §6).

In [ ]:
data = SaundersAdapter(SLICE_DIR, max_files=MAX_FILES, counts_layer=COUNTS_LAYER).load()
n_pert = len(data.perturbations())
print(f'{data.n_cells} cells, {data.n_genes} genes, {n_pert} perturbations, '
      f'{int(data.is_control.sum())} control cells, {int(data.is_perturbed.sum())} perturbed cells')

# Keep only EVAL perturbations actually present in this slice (defensive).
present = set(data.perturbations())
EVAL = [p for p in EVAL if p in present]
print('evaluating', len(EVAL), 'perturbations present in this slice:', EVAL)
assert EVAL, 'none of the EVAL perturbations are present in this slice — check SLICE_DIR / EVAL'

## 3. Niche graph + cell-type composition (D3 input)
Two interchangeable spatial graphs, both returning a `(2, n_edges)` `[src, dst]` array built WITHIN each slice (no cross-batch edges, no self-loops):
- `build_knn_graph(data, k)` — pure sklearn kNN (the harness/`run_benchmark` default).
- `build_spatial_graph(data, n_neighs)` — squidpy-backed (optional; needs squidpy installed).

`compute_niche_composition(data, edges, hops)` then gives a `(n_cells, C)` row-simplex of each cell's neighbourhood cell-type composition — the **D3** quantity. We build the kNN graph here and reuse it everywhere below so the niche definition is consistent.

In [ ]:
edges = build_knn_graph(data, k=K_GRAPH)
print('kNN graph:', edges.shape[1], 'directed edges')

# Optional squidpy graph (same (2, n_edges) format); falls back to the kNN graph if squidpy
# is not installed in this env.
try:
    sq_edges = build_spatial_graph(data, n_neighs=K_GRAPH)
    print('squidpy spatial graph:', sq_edges.shape[1], 'edges (drop-in alternative to kNN)')
except Exception as e:
    print('squidpy graph unavailable (', type(e).__name__, '), using kNN graph only')

comp = compute_niche_composition(data, edges, hops=NICHE_HOPS)   # (n_cells, C) D3 row-simplex
C = comp.shape[1]
print('niche composition (D3):', comp.shape, '->', C, 'cell types; row sums ~',
      float(np.nanmean(comp.sum(1)[comp.sum(1) > 0])))

### How a model's output becomes a `StandardPrediction` (the output adapter)
`to_prediction(seed_out, prop_out, composition)` maps a model's native arrays onto the `{D1, D2, D3}` contract: it mean-aggregates per-row seed/prop outputs to a single gene vector and L1-normalises the composition to a simplex. `.covered_dims()` / `.as_dict()` then report exactly which dims are filled — this is how the capability matrix is enforced per model.

In [ ]:
# Illustration on one perturbation's cells (no scoring here — just the contract).
p0 = EVAL[0]
centers0 = np.where(data.perturbation == p0)[0]
seed_out0 = data.X[centers0]                         # stand-in seed rows (m, G)
comp0 = comp[centers0].mean(0)                       # mean neighbourhood composition (C,)

pred_seed_only = to_prediction(seed_out=seed_out0)                  # a D1-only model (e.g. scGEN)
pred_full      = to_prediction(seed_out=seed_out0, prop_out=seed_out0, composition=comp0)
print(p0, 'seed-only model covers :', pred_seed_only.covered_dims())
print(p0, 'full   model covers    :', pred_full.covered_dims())
print('D3 is a simplex, sums to   :', round(float(pred_full.d3.sum()), 6))

## 4. MC-spatial quadrants (optional)
MC-spatial writes a `*_Dual_Metrics.csv` per slice with two axis p-values per guide: `p_val_specificity` (X = a real **Self/D1** effect) and `p_val_global` (Y = a real **Niche/D2** effect). `mc_spatial_join.load_dual_metrics(csv, p_cutoff, p_mode)` reads it (stdlib `csv`, no pandas) and tags each perturbation with its quadrant — `BOTH`, `X_ONLY`, `Y_ONLY`, `INERT` (label constants live in `mc_spatial_join`). If `DUAL_METRICS_CSV` is unset or missing, we fall back to the unstratified EVAL list and clearly note that MC-spatial was not provided.

In [ ]:
QUADRANT_OF = {}        # perturbation -> quadrant label (empty => no MC-spatial labels)
HAVE_MC = bool(DUAL_METRICS_CSV) and Path(DUAL_METRICS_CSV).exists()
if HAVE_MC:
    dual = mcj.load_dual_metrics(DUAL_METRICS_CSV, p_cutoff=0.05, p_mode='raw')
    QUADRANT_OF = {d['perturbation']: d['quadrant'] for d in dual}
    from collections import Counter
    counts = Counter(QUADRANT_OF.get(p, mcj.UNKNOWN) for p in EVAL)
    print('MC-spatial quadrants for the EVAL guides:')
    for q in (mcj.BOTH, mcj.X_ONLY, mcj.Y_ONLY, mcj.INERT, mcj.UNKNOWN):
        if counts.get(q):
            print(f'  {q:18s}: {counts[q]}')
    print('  (Inert is the negative control; D1 is scored on Self/Both, D2 on Niche/Both.)')
else:
    print('NOTE: MC-spatial Dual_Metrics.csv not provided (DUAL_METRICS_CSV is unset/missing).')
    print('      -> running UNSTRATIFIED on the EVAL list; the §7 stratified report is skipped.')

## 5. Run the baselines (D1 seed + D2 niche)
`run_benchmark` now uses the **sample-level aggregate control** by default (`_control_reference_aggregate`) — no matched-control feature pairing. For each perturbation it composes a **TrivialSeed** (D1 floor), a **Gaussian-kernel** baseline propagation and a self-supervised **GCN** learned propagation (both D2), and scores them. It returns:
- `res['seed'][p]` — the **D1** seed score (`pcc_delta` = direction of the cell's own shift, `mse` = magnitude).
- `res['compare'][p]` — the **D2** niche score (`gain`/`pcc`/`e` per 2x2 method; the deployable cell is `model+learned`). `gain = e_null - e` > 0 beats the no-effect baseline, where **e_null is the aggregate same-type control niche** (the neighbourhood of the average unperturbed cell of that type) — NOT a matched or permutation null.

In [ ]:
res = run_benchmark(data, perturbations=EVAL, k=K_GRAPH, k_ref=K_REF,
                    gcn_kwargs={'hidden': 64, 'epochs': 30})

rows = []
for p in EVAL:
    s = res['seed'][p]; c = res['compare'][p]
    rows.append(dict(
        perturbation=p,
        quadrant=QUADRANT_OF.get(p, mcj.UNKNOWN),
        d1_trivial_pcc=s['pcc_delta'], d1_trivial_mse=s['mse'],    # D1 from TrivialSeed
        d2_gain=c['gain']['model+learned'], d2_pcc=c['pcc']['model+learned'],  # D2 deployable
        d2_gain_oracle=c['gain'].get('oracle'), e_null=c['e']['null'],
        leak_ok=res['leakage_pass'][p],
    ))
base_df = pd.DataFrame(rows).sort_values('d2_gain', ascending=False).reset_index(drop=True)
base_df

## 5b. Seed / niche method-comparison box plots (the headline view)
Mirrors the CONCERT-style figure: **one box per method**, where the box = that method's pooled **per-repeat matched-n energy distance** (`res[...]['e_samples']`). **Left = seed (D1)**, **right = niche (D2)**; the two dashed lines are the no-effect `null` and the `oracle` ceiling. The learned-prop method is named **GCN** on the niche axis (Gaussian = the baseline prop). Lower energy = closer to the observed perturbed cells; a box sitting **below `null`** beats predicting 'no effect'. This is `plot_seed_prop(res)` — the per-dataset headline plot.

In [ ]:
from spbench.plotting import plot_seed_prop, collect_seed_samples, collect_prop_samples
fig = plot_seed_prop(res)
fig.savefig('seed_prop_methods.png', dpi=130, bbox_inches='tight')
_sb, _ = collect_seed_samples(res); _pb, _ = collect_prop_samples(res)
print('seed axis methods :', list(_sb))
print('niche axis methods:', list(_pb), '(GCN = learned prop)')
fig

## 6. scGEN — a D1-only conditional model (optional)
scGEN is run OFFLINE in its own env; per perturbation it dumps `{P}_seed.h5ad` (a per-center-aligned predicted-seed array). `ScgenSeedModel({P: path})` loads it and serves it as the model seed; `.centers(P)` gives the StandardData center indices the rows align to. We score scGEN's **D1** in its own log-norm space: build `lognorm_X = build_lognorm_X(data)` (raw counts -> normalize_total 1e4 + log1p) and pass `eval_X=lognorm_X` into `fill_2x2`. `fill_2x2` then slices the observed/reference seed cells into that same matrix (the three are co-spaced) and carries `eval_X=None` downstream, so `evaluate_seed(niches)` scores fairly.

If `SCGEN_DUMP_DIR` has no `{P}_seed.h5ad` for any EVAL perturbation, this cell is a clearly marked no-op and the notebook stays baseline-only.

In [ ]:
from spbench.models.scgen_model import ScgenSeedModel

scgen_paths = {}
if SCGEN_DUMP_DIR:
    for p in EVAL:
        f = Path(SCGEN_DUMP_DIR) / f'{p}_seed.h5ad'
        if f.exists():
            scgen_paths[p] = str(f)

scgen_d1 = {}        # perturbation -> {'pcc_delta','mse','n'} for scGEN's D1
if scgen_paths:
    print(f'scGEN dumps found for {len(scgen_paths)}/{len(EVAL)} perturbations.')
    # scGEN's seed_pred lives in the log-norm space -> score there via eval_X=lognorm_X.
    lognorm_X = build_lognorm_X(data)                  # (n_cells, n_genes) log-norm matrix
    X_ref_agg = _control_reference_aggregate(data, edges)   # aggregate-control reference (G1)
    resid = _control_residuals(data)
    base_prop = GaussianProp().fit(data, edges)
    learned_prop = SimpleGCN(hidden=64, epochs=30).fit(data, edges)
    for p, path in scgen_paths.items():
        seed_model = ScgenSeedModel({p: path})         # offline loader; predict_seed is 2-arg
        grid = fill_2x2(data, p, edges, seed_model, base_prop, learned_prop, k_ref=K_REF,
                        X_ref=X_ref_agg, return_niches=True, residuals=resid,
                        eval_X=lognorm_X)              # log-norm scoring space for D1
        niches = grid['_niches']                        # eval_X already consumed into the matrix
        scgen_d1[p] = evaluate_seed(niches, eval_X=niches.get('eval_X'))   # carried eval_X is None here
    sc_df = pd.DataFrame([dict(perturbation=p, quadrant=QUADRANT_OF.get(p, mcj.UNKNOWN),
                               scgen_d1_pcc=v['pcc_delta'], scgen_d1_mse=v['mse'])
                          for p, v in scgen_d1.items()])
    base_df = base_df.merge(sc_df.drop(columns='quadrant'), on='perturbation', how='left')
    display(sc_df.sort_values('scgen_d1_pcc', ascending=False).reset_index(drop=True))
else:
    print('NOTE: no scGEN {P}_seed.h5ad dumps found (SCGEN_DUMP_DIR unset/empty).')
    print('      -> running BASELINE-ONLY; scGEN D1 columns are omitted.')

### Combined per-perturbation results table
One row per perturbation: the baselines' D1 (TrivialSeed `pcc_delta`/`mse`) and D2 (`model+learned` gain/pcc), plus scGEN's D1 when its dumps were found. This is the per-perturbation evidence behind the stratified capability matrix in §7.

In [ ]:
base_df

## 7. Capability matrix + stratified report
We join the MC-spatial quadrant onto each model's per-dimension scores and report, **per dimension, only over the quadrants where that dimension has signal**, with **Inert as the negative control**:
- **D1** (self) — scored on **X-Only + Both** (`mc_spatial_report.DIMENSION_QUADRANTS['d1']`).
- **D2** (niche) — scored on **Y-Only + Both** (`DIMENSION_QUADRANTS['d2']`).
- **Inert** — negative control: a good model has `mean_gain ~ 0` there; a positive gain in Inert is a hallucinated effect.

`mc_spatial_report.stratified_report(records, dim_gain_fields)` consumes records that carry a `quadrant` label (from `join_quadrants`) plus per-dimension **gain** fields and returns the `dimension x quadrant_group` rows (`mean_gain`, `frac_beat`, `is_negative_control`). We score D1 via its own gain (TrivialSeed D1 vs scGEN D1) and D2 via the niche gain. If MC-spatial was not provided, we show an unstratified capability matrix instead.

In [ ]:
def _capability_matrix(df):
    """Which model covers which dimension (mean score over the rows we have)."""
    cm = {'TrivialSeed': {'D1': float(np.nanmean(df['d1_trivial_pcc'])), 'D2': np.nan, 'D3': np.nan},
          'Gaussian/GCN (model+learned)': {'D1': np.nan,
                                           'D2': float(np.nanmean(df['d2_gain'])), 'D3': np.nan},
          'niche-composition module': {'D1': np.nan, 'D2': np.nan, 'D3': 'simplex (built in §3)'}}
    if 'scgen_d1_pcc' in df.columns:
        cm['scGEN'] = {'D1': float(np.nanmean(df['scgen_d1_pcc'])), 'D2': np.nan, 'D3': np.nan}
    return pd.DataFrame(cm).T

print('Capability matrix (mean over EVAL; D1=pcc_delta, D2=niche gain; NaN = dim not covered):')
display(_capability_matrix(base_df))

In [ ]:
if HAVE_MC:
    # Per-perturbation D1 gain = scGEN D1 pcc - TrivialSeed D1 pcc (how much scGEN beats the
    # floor on the cell's own shift); D2 gain = the niche gain over the no-effect baseline.
    records = []
    for _, r in base_df.iterrows():
        d1_gain = (r['scgen_d1_pcc'] - r['d1_trivial_pcc']) if 'scgen_d1_pcc' in base_df.columns \
            and pd.notna(r.get('scgen_d1_pcc')) else r['d1_trivial_pcc']
        records.append({'perturbation': r['perturbation'],
                        'gain_d1': float(d1_gain), 'gain_d2': float(r['d2_gain'])})
    # Attach quadrants from the CSV (left join; absent guides -> 'Unknown').
    records = mcj.join_quadrants(records, DUAL_METRICS_CSV, key='perturbation', p_mode='raw')
    report = mcr.stratified_report(records, dim_gain_fields={'d1': 'gain_d1', 'd2': 'gain_d2'})
    rep_df = pd.DataFrame(report)
    print('Stratified report (D1 on Self/Both, D2 on Niche/Both, Inert = negative control):')
    display(rep_df)
    print('Read: mean_gain should be > 0 in the signal rows and ~ 0 in the is_negative_control'
          ' (Inert) rows.')
else:
    print('MC-spatial not provided -> stratified report skipped. Unstratified capability'
          ' matrix above is the summary; rerun with DUAL_METRICS_CSV set for the quadrant'
          ' x dimension breakdown with the Inert negative control.')

## 8. Bottom line — how to read it
- **Capability matrix.** Each model is scored only on the dims it covers: scGEN on **D1**, the Gaussian/GCN propagation on **D2**, the niche module supplies **D3**. NaN means 'dim not covered', not 'failed'.
- **scGEN vs TrivialSeed on D1.** scGEN is a conditional model; it should **beat TrivialSeed on Self/Both** (positive `gain_d1`) and sit **≈ the baseline on Inert** (`mean_gain ~ 0` in the negative-control row). A large positive D1 gain in the Inert row would mean scGEN is hallucinating an effect where MC-spatial found none.
- **D2 niche gain.** `model+learned` `gain > 0` beats predicting 'the neighbours did not change', and it should be concentrated in the **Niche/Both** quadrants, not in Inert.
- **eval_X.** scGEN's D1 is scored in its log-norm space; the baselines' D1 is in `data.X` space — compare each model to its own control delta within the stratified report, not the raw numbers across spaces.

In [ ]:
print('Per-dimension coverage this run:')
print('  D1 (self)        : TrivialSeed' + (', scGEN' if 'scgen_d1_pcc' in base_df.columns else '')
      + '  (scored on Self/Both quadrants)')
print('  D2 (niche expr)  : Gaussian + GCN (model+learned)  (scored on Niche/Both quadrants)')
print('  D3 (niche comp)  : compute_niche_composition simplex, shape', comp.shape)
n_d2_win = int((base_df['d2_gain'] > 0).sum())
print(f'\nD2: {n_d2_win}/{len(base_df)} perturbations beat the no-effect baseline (gain>0).')
if 'scgen_d1_pcc' in base_df.columns:
    better = base_df['scgen_d1_pcc'] > base_df['d1_trivial_pcc']
    print(f'D1: scGEN beats TrivialSeed on {int(better.sum())}/{int(better.notna().sum())} '
          'perturbations (pcc_delta).')
if not HAVE_MC:
    print('\n(MC-spatial labels not provided -> the Inert negative control was not evaluated.)')

## 9. Visual summary (the headline result)
Three panels over this slice's guides plus a clean text dump of the final table:
- **D1** — scGEN vs the TrivialSeed floor (`pcc_delta`, higher is better): is the conditional model beating the floor on the cell's own shift?
- **D2** — niche gain vs the oracle upper bound (`gain = e_null - e`, >0 beats predicting 'neighbours unchanged').
- **Capability matrix** heatmap — each model scored only on the dims it covers ('—' = not covered).
The inline figure is also saved next to the notebook as `run_benchmark_results.png`.

In [ ]:
import matplotlib.pyplot as plt

has_scgen = 'scgen_d1_pcc' in base_df.columns
g = base_df['perturbation'].tolist()
x = np.arange(len(g)); w = 0.38
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

# (1) D1 — scGEN vs TrivialSeed floor
ax = axes[0]
ax.bar(x - w/2, base_df['d1_trivial_pcc'], w, label='TrivialSeed (floor)', color='#b8b8b8')
if has_scgen:
    ax.bar(x + w/2, base_df['scgen_d1_pcc'], w, label='scGEN', color='#2c7fb8')
ax.axhline(0, color='k', lw=.6)
ax.set_xticks(x); ax.set_xticklabels(g, rotation=30, ha='right')
ax.set_ylabel('D1  pcc_delta  (higher = better)')
ax.set_title('D1 - self shift: scGEN vs floor')
ax.legend(fontsize=8, loc='upper right')

# (2) D2 - niche gain vs oracle upper bound
ax = axes[1]
ax.bar(x - w/2, base_df['d2_gain'], w, label='model+learned', color='#41ab5d')
if 'd2_gain_oracle' in base_df.columns:
    oracle = pd.to_numeric(base_df['d2_gain_oracle'], errors='coerce')
    ax.bar(x + w/2, oracle, w, label='oracle (upper bound)', color='#cfcfcf')
ax.axhline(0, color='k', lw=.8)
ax.set_xticks(x); ax.set_xticklabels(g, rotation=30, ha='right')
ax.set_ylabel('D2 niche gain = e_null - e  (>0 beats no-effect)')
ax.set_title('D2 - niche gain (Self-sig guides -> weak niche)')
ax.legend(fontsize=8, loc='lower left')

# (3) capability matrix heatmap
ax = axes[2]
cm = _capability_matrix(base_df)
num = cm[['D1', 'D2']].apply(pd.to_numeric, errors='coerce').astype(float)
im = ax.imshow(num.values, aspect='auto', cmap='RdYlBu_r', vmin=-0.2, vmax=0.6)
ax.set_xticks(range(num.shape[1])); ax.set_xticklabels(num.columns)
ax.set_yticks(range(num.shape[0])); ax.set_yticklabels(num.index, fontsize=7.5)
for i in range(num.shape[0]):
    for j in range(num.shape[1]):
        v = num.values[i, j]
        ax.text(j, i, '-' if np.isnan(v) else f'{v:.3f}', ha='center', va='center', fontsize=9)
ax.set_title('Capability matrix (- = not covered)')
fig.colorbar(im, ax=ax, fraction=.046, pad=.04)

fig.suptitle('Saunders Batch_10_Slice_0  -  ' + str(len(g)) + ' guides  -  D1 / D2 / D3', y=1.04)
fig.tight_layout()
fig.savefig('run_benchmark_results.png', dpi=130, bbox_inches='tight')
plt.show()
print('saved figure -> notebooks/run_benchmark_results.png')

In [ ]:
# ---- clean text dump of the final results ----
pd.set_option('display.float_format', lambda v: f'{v:.4f}')
cols = ['perturbation', 'quadrant', 'd1_trivial_pcc']
if has_scgen:
    cols += ['scgen_d1_pcc']
cols += ['d2_gain', 'd2_pcc']
print('=' * 72)
print('FINAL RESULTS  -  Saunders Batch_10_Slice_0  (one row per guide)')
print('=' * 72)
print(base_df[cols].to_string(index=False))
print('\nCapability matrix (mean over guides; - = dimension not covered by that model):')
print(_capability_matrix(base_df).to_string())
if has_scgen:
    sc_mean = base_df['scgen_d1_pcc'].mean(); tr_mean = base_df['d1_trivial_pcc'].mean()
    n_win = int((base_df['scgen_d1_pcc'] > base_df['d1_trivial_pcc']).sum())
    print(f'\nD1: scGEN mean {sc_mean:.3f}  vs  TrivialSeed mean {tr_mean:.3f}'
          f'   (scGEN wins {n_win}/{len(base_df)} guides)')
n_d2 = int((base_df['d2_gain'] > 0).sum()); d2_mean = base_df['d2_gain'].mean()
print(f'D2: niche gain > 0 on {n_d2}/{len(base_df)} guides  (mean gain {d2_mean:.3f})')